# 02 · Retrain (v1.1 protocol) + external test on NIH ChestX-ray14

Run this on **Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU). It takes about 5–6 hours in total, mostly training.

What it does, in order:
1. Retrains DenseNet121 with the **leak-free protocol**: the best epoch is picked on a patient-level split of train, not on the validation set.
2. Logs the run to **MLflow**, saved in your Google Drive.
3. Evaluates once on the CheXpert validation set, with 95% confidence intervals.
4. Runs the **quality gate** against the old v1.0 numbers.
5. Builds the **drift reference** for the API.
6. Runs the **batch drift check + external validation on a different dataset** (NIH ChestX-ray14, from a different hospital).

Everything is written to `MyDrive/chexpert_v1_4/`, so nothing is lost if Colab disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/chexpert_v1_4'
import os; os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
os.environ['MLFLOW_TRACKING_URI'] = f'file:{OUT}/mlruns'

In [ ]:
!git clone -q https://github.com/AtomicHalifax/ChestXRay-Disease-Classification.git /content/repo
%cd /content/repo
!pip install -q -e ".[track]" kagglehub
!git log --oneline -1

## 1. Get CheXpert

**Option A (fastest):** reuse the zip already in your Drive from the v1.0 notebook (`MyDrive/ChestXRay Data.zip`).
**Option B:** download it from Kaggle with `kagglehub`. It asks you to log in to Kaggle once.

In [ ]:
import os, glob
ZIP = '/content/drive/MyDrive/ChestXRay Data.zip'
if os.path.exists(ZIP):
    !unzip -q -o "{ZIP}" -d /content/dataset
    DATA = '/content/dataset'
else:
    import kagglehub
    path = kagglehub.dataset_download('ashery/chexpert')
    DATA = os.path.dirname(glob.glob(f'{path}/**/train.csv', recursive=True)[0])
os.environ['DATA'] = DATA
print('CheXpert at', DATA, os.listdir(DATA)[:6])

## 2. Train (≈4.5 h)

Checks `train.csv` first (data validation), then trains for 10 epochs and keeps the epoch with the best AUROC on the **5% patient-level tuning split**. Each epoch is logged to MLflow.

In [ ]:
!python -m chexpert_cls.train --data-root "$DATA" --out "$OUT/run" \
    --epochs 10 --pos-weight --workers 4 --mlflow --experiment chexpert-densenet121

## 3. Evaluate once on the CheXpert validation set

In [ ]:
!python -m chexpert_cls.evaluate --data-root "$DATA" --weights "$OUT/run/best.pth" --out "$OUT/results"

## 4. Quality gate vs the v1.0 baseline

The v1.0 numbers were **optimistic**, because that checkpoint was picked on this same validation set. A fair new model can come out slightly lower and fail the gate. That's the gate doing its job.

If it fails only by a small margin that sits inside the confidence intervals, the honest move is to adopt the new model anyway and replace the baseline. Write down why in the README.

In [ ]:
!python -m chexpert_cls.gate --candidate "$OUT/results/valid_report.json" || echo "GATE FAILED (see table above)"

## 5. Build the drift reference (switches on drift detection in the API)

In [ ]:
!python -m chexpert_cls.drift --predictions "$OUT/results/valid_predictions.npz" --out "$OUT/drift_reference.json"

## 6. A *new* dataset: NIH ChestX-ray14 (external validation + batch drift check)

This is a different hospital (NIH Clinical Center) with different scanners and a different labeller, which makes it a realistic "someone hands you new data" test. It uses the Kaggle 5% sample (about 5,600 images). All five of our findings exist in NIH; their "Effusion" label maps to our "Pleural Effusion".

What to expect: AUROC usually **drops** on external data, and drift will probably show a shift. Both are normal findings and worth reporting, not hiding.

In [ ]:
import kagglehub, glob, os
nih = kagglehub.dataset_download('nih-chest-xrays/sample')
labels = glob.glob(f'{nih}/**/sample_labels.csv', recursive=True)[0]
images = os.path.dirname(glob.glob(f'{nih}/**/images/*.png', recursive=True)[0])
os.environ['NIH_LABELS'], os.environ['NIH_IMAGES'] = labels, images
print(len(os.listdir(images)), 'images;', labels)

In [ ]:
!python -m chexpert_cls.drift_check "$NIH_IMAGES" --nih-labels "$NIH_LABELS" \
    --weights "$OUT/run/best.pth" --reference "$OUT/drift_reference.json" --out "$OUT/nih_report"

## 7. Save the fingerprint, then bring the results back to GitHub

1. Note the SHA-256 printed below. It goes in `CHEXPERT_WEIGHTS_SHA256`.
2. From `MyDrive/chexpert_v1_4/`, add these to the repo:
   - `results/valid_report.json` → `models/candidate_metrics.json` (CI runs the quality gate on it)
   - `drift_reference.json` → `models/drift_reference.json`
   - `nih_report/report.md` → `docs/EXTERNAL_VALIDATION.md`
3. Upload the new weights to Hugging Face (cell below; needs a write token), then bump `MODEL_VERSION` in `src/chexpert_cls/config.py`.

In [ ]:
import hashlib
h = hashlib.sha256(open(f"{OUT}/run/best.pth", 'rb').read()).hexdigest()
print('SHA-256 of new weights:', h)

In [ ]:
# Optional: publish the new weights to Hugging Face as a new commit of the same repo.
# from huggingface_hub import login, upload_file
# login()  # paste a token with write access
# upload_file(path_or_fileobj=f"{OUT}/run/best.pth", path_in_repo="best_densenet121.pth",
#             repo_id="AtomicHalifax/ChestXRay-DenseNet121", commit_message="v1.1 protocol retrain")